In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
import os

In [2]:
data = pd.read_parquet("../data/features/data_with_scores.parquet")
print(f"Загружено: {data.shape}")

Загружено: (368014, 69)


In [3]:
# Нормализация ensemble_score → risk_score (0-100)
scaler = MinMaxScaler(feature_range=(0, 100))
data["risk_score"] = scaler.fit_transform(
    data[["ensemble_score"]]
).flatten().round(1)

print(data["risk_score"].describe())

count    368014.00000
mean         12.80305
std           9.42832
min           0.00000
25%           5.20000
50%          10.70000
75%          17.90000
max         100.00000
Name: risk_score, dtype: float64


In [4]:
# Бустеры — берём максимальный коэффициент, не перемножаем
suspicious_pairs = ["71.01_50.02", "60.01_91.01", "76.09_76.09"]

def get_boost(row):
    boosts = []
    if row["is_manual"] == 1:
        boosts.append(1.5)
    if row["is_amount_outlier"] == 1:
        boosts.append(1.3)
    if row["is_night"] == 1:
        boosts.append(1.3)
    if row["is_first_operation"] == 1:
        boosts.append(1.2)
    if row["account_pair"] in suspicious_pairs:
        boosts.append(1.5)
    return max(boosts) if boosts else 1.0

data["boosted_score"] = (
    data.apply(get_boost, axis=1) * data["risk_score"]
).clip(0, 100).round(1)

In [5]:
data.columns

Index(['period', 'Регистратор', 'contractor', 'КонтрагентИНН', 'account_dt',
       'СубконтоДт1', 'ВидСубконтоДт1', 'СубконтоДт2', 'ВидСубконтоДт2',
       'СубконтоДт3', 'ВидСубконтоДт3', 'account_kt', 'СубконтоКт1',
       'ВидСубконтоКт1', 'СубконтоКт2', 'ВидСубконтоКт2', 'СубконтоКт3',
       'ВидСубконтоКт3', 'dept_dt', 'dept_kt', 'amount', 'Содержание',
       'ТипДокумента', 'is_manual', 'hour', 'day_of_week', 'month',
       'is_weekend', 'is_night', 'is_mounth_end', 'is_quarter_end',
       'log_amount', 'is_negative_amount', 'abs_amount', 'account_pair',
       'pair_frequency', 'log_pair_frequency', 'is_rare_pair', 'pair_mean',
       'pair_std', 'amount_zscore', 'is_amount_outlier', 'contractor_freq',
       'log_contractor_frequency', 'contractor_ops_before',
       'is_first_operation', 'dept_dt_freq', 'dept_kt_freq', 'large_and_rare',
       'late_and_large', 'new_cont_and_large', 'manual_and_large',
       'first_contractor_pair', 'is_first_contractor_pair', 'date',
  

In [6]:
# Whitelist — обнуляем только если is_manual = 0
whitelist_doc_types = [
    "Регламентная операция",
    "Отражение зарплаты в бухучете",
    "Операции ЕНС",
    "Уведомление об исчисленных суммах налогов",
    "Сведения об удержанном НДФЛ",
    "Закрытие месяца",
    "Амортизация",
    "Переоценка валюты",
    "Начисление зарплаты",
    "Поступление наличных",
]

whitelist_mask = (
    data["ТипДокумента"].isin(whitelist_doc_types) & 
    (data["is_manual"] == 0)
)

tax_payment_mask = (
    (data["account_pair"] == "68.90_51") &
    (data["ТипДокумента"] == "Списание с расчетного счета") &
    (data["is_manual"] == 0)
)


# Безусловный whitelist — дивиденды если не ручная
dividends_mask = (
    (data["account_pair"] == "84.01_51") &
    (data["ТипДокумента"] == "Списание с расчетного счета") &
    (data["is_manual"] == 0)
)

auto_cost_mask = (
    (data["account_pair"] == "90.02.1_41.01") &
    (data["ТипДокумента"] == "Реализация (акт, накладная, УПД)") &
    (data["is_manual"] == 0)
)

bank_commission_mask = (
    (data["account_pair"] == "91.02_51") &
    (data["ТипДокумента"] == "Списание с расчетного счета") &
    (data["is_manual"] == 0)
)


ens_mask = (
    (data["account_pair"] == "68.12_68.90") &
    (data["ТипДокумента"] == "Операция по единому налоговому счету") &
    (data["is_manual"] == 0)
)

incassation_mask = (
    (data["account_pair"] == "51_57.01") &
    (data["ТипДокумента"] == "Поступление на расчетный счет") &
    (data["is_manual"] == 0)
)

deposit_mask = (
    (data["account_pair"] == "51_55.03") &
    (data["ТипДокумента"] == "Поступление на расчетный счет") &
    (data["is_manual"] == 0)
)

data.loc[
    whitelist_mask | dividends_mask | tax_payment_mask | 
    auto_cost_mask | bank_commission_mask | 
    ens_mask | incassation_mask | deposit_mask,
    "boosted_score"
] = 0

data.loc[whitelist_mask | dividends_mask | tax_payment_mask | auto_cost_mask | bank_commission_mask, "boosted_score"] = 0

data.loc[whitelist_mask | dividends_mask | tax_payment_mask, "boosted_score"] = 0
data.loc[whitelist_mask | dividends_mask, "boosted_score"] = 0

print(f"Обнулено через whitelist: {(whitelist_mask | dividends_mask).sum()} строк")
print(f"boosted_score > 0: {(data['boosted_score'] > 0).sum()} строк")

Обнулено через whitelist: 12271 строк
boosted_score > 0: 232582 строк


In [7]:
# Функция объяснений
def explain_anomaly(row):
    reasons = []
    if row["is_rare_pair"]:
        reasons.append(f"Редкая пара счетов {row['account_dt']}→{row['account_kt']}")
    if row["is_amount_outlier"]:
        reasons.append("Сумма необычно большая для этого типа операции")
    if row["is_night"]:
        reasons.append(f"Операция в нерабочее время ({row['hour']}:00)")
    if row["is_weekend"]:
        reasons.append("Операция в выходной день")
    if row["is_first_operation"]:
        reasons.append("Первая операция с этим контрагентом")
    if row["is_manual"]:
        reasons.append("Ручная проводка")
    if row["is_negative_amount"]:
        reasons.append("Сторно (отрицательная сумма)")
    if row["account_pair"] in ["71.01_50.02", "60.01_91.01"]:
        reasons.append("Подозрительная пара счетов")
    if not reasons:
        reasons.append("Комбинация факторов")
    return "; ".join(reasons)

data["explanation"] = data.apply(explain_anomaly, axis=1)
print("Объяснения добавлены")

Объяснения добавлены


In [8]:
# Топ-35 для отчёта
report_data = (
    data[data["abs_amount"] >= 100]
    .nlargest(35, "boosted_score")
    [[
        "period", "account_dt", "account_kt", "amount", "pair_mean",
        "contractor", "Содержание", "ТипДокумента", "boosted_score", "explanation"
    ]]
    .copy()
)

report_data.columns = [
    "Дата", "Счет Дт", "Счет Кт", "Сумма", "Средняя сумма по паре",
    "Контрагент", "Содержание", "Тип документа", "Риск (0-100)", "Причина"
]
report_data["Риск (0-100)"] = report_data["Риск (0-100)"].round(1)
report_data["Средняя сумма по паре"] = report_data["Средняя сумма по паре"].round(2)

os.makedirs("../reports", exist_ok=True)
report_data.to_excel("../reports/anomalies_v3.xlsx", index=False, sheet_name="Аномалии")

# Форматирование
wb = openpyxl.load_workbook("../reports/anomalies_v3.xlsx")
ws = wb["Аномалии"]

red_fill    = PatternFill(start_color="FFCCCC", end_color="FFCCCC", fill_type="solid")
yellow_fill = PatternFill(start_color="FFFFCC", end_color="FFFFCC", fill_type="solid")
header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center")

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    risk = row[8].value  # колонка "Риск (0-100)"
    if risk and risk > 80:
        for cell in row:
            cell.fill = red_fill
    elif risk and risk > 60:
        for cell in row:
            cell.fill = yellow_fill

# Колонки для разметки бухгалтером
ws.cell(row=1, column=11, value="Оценка").fill = header_fill
ws.cell(row=1, column=11).font = header_font
ws.cell(row=1, column=12, value="Комментарий").fill = header_fill
ws.cell(row=1, column=12).font = header_font

dv = DataValidation(
    type="list",
    formula1='"Ошибка,Норма,Непонятно"',
    allow_blank=True,
    showDropDown=False,
)
ws.add_data_validation(dv)
dv.sqref = f"K2:K{ws.max_row}"

for col in ws.columns:
    max_length = max(len(str(cell.value or "")) for cell in col)
    ws.column_dimensions[col[0].column_letter].width = min(max_length + 2, 50)

ws.column_dimensions["K"].width = 15
ws.column_dimensions["L"].width = 40

wb.save("../reports/anomalies_v3.xlsx")
print("Отчёт сохранён: reports/anomalies_v3.xlsx")

Отчёт сохранён: reports/anomalies_v3.xlsx
